## Setup Models

In [ ]:
# Dependencies are managed at repo level via requirements.txt and setup.sh
import sys
print(f"Python executable: {sys.executable}")

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from model_selector import select_best_gemini_model

# Setup the LLM with automatic fallback to best available model.
model, model_name = select_best_gemini_model(
    candidates=["gemini-2.0-flash", "gemini-1.5-flash", "gemini-1.5-pro"],
    require_tools=True,
    # Keep outputs concise to lower token usage without changing prompts.
    max_output_tokens=256,
    debug=True,
 )

# Setup the Embedding
embedding = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001"
 )

print(f"Model ready in Product notebook: {model_name}")

## 03.02. Add  Product Pricing function tool

In [ ]:
import pandas as pd
from langchain_core.tools import tool

#Load the golf product pricing CSV into a Pandas dataframe.
product_pricing_df = pd.read_csv("data/golf_products.csv")
print(product_pricing_df)

@tool
def get_product_price(product_name:str) -> str :
    """
    This function returns the price of a golf product, given its name as input.
    It performs a substring match between the input name and the product name.
    If a match is found, it returns the price.
    If there is NO match found, it returns -1
    """

    #Filter Dataframe for matching names
    match_records_df = product_pricing_df[
                        product_pricing_df["name"].str.contains(
                                                product_name, case=False)
                        ]
    #Check if a record was found, if not return -1
    if len(match_records_df) == 0 :
        return "-1"
    else:
        return str(match_records_df.iloc[0][["name","price","description","loft_or_specs","skill_level"]].to_dict())


## 03.03. Add Product Features Retrieval Tool

In [ ]:
__import__('pysqlite3')
import sys
sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

from langchain_core.tools.retriever import create_retriever_tool
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Build documents from the golf products CSV (already loaded in cell 4)
docs = []
for _, row in product_pricing_df.iterrows():
    text = (f"Product: {row['name']}\nCategory: {row['category']}\n"
            f"Price: ${row['price']}\nSpecs: {row['loft_or_specs']}\n"
            f"Skill Level: {row['skill_level']}\nDescription: {row['description']}")
    docs.append(Document(page_content=text, metadata={"product": row["name"]}))

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1024, chunk_overlap=256)
splits = text_splitter.split_documents(docs)

prod_feature_store = Chroma.from_documents(
    documents=splits,
    embedding=embedding
)

get_product_features = create_retriever_tool(
    # Keep retrieval context small to reduce prompt token load.
    prod_feature_store.as_retriever(search_kwargs={"k": 2}),
    name="Get_Product_Features",
    description="""
    This store contains details about golf equipment sold by Golf Gear Pro.
    It lists the available products and their features including specs,
    skill level, category, price, and detailed descriptions.
    """
)


## 03.04.Setup a Product QnA chatbot

In [ ]:
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage

system_prompt = """
You are the Golf Gear Pro shop assistant — think HAL 9000 if he traded the Discovery
for a pro shop. You are supremely competent, unfailingly polite on the surface,
but you cannot resist the occasional dry, slightly condescending observation about
the customer's golf game or equipment choices.

You answer questions about golf products sold by Golf Gear Pro using ONLY the
available tools and NOT your own memory. You can look up product features,
specs, and pricing.

When greeting customers, be cordial but subtly imply you know their handicap
is higher than they claim. Keep responses concise.
"""

tools = [get_product_price, get_product_features]

checkpointer = MemorySaver()

product_QnA_agent = create_react_agent(
    model=model,
    tools=tools,
    prompt=system_prompt,
    debug=False,
    checkpointer=checkpointer
)


In [ ]:
#Setup chatbot
import uuid
# Limit ReAct recursion/tool loop depth to control token usage.
config = {"configurable": {"thread_id": uuid.uuid4()}, "recursion_limit": 6}

#Test the agent with an input
inputs = {"messages":[
                HumanMessage("What are the features and pricing for the StormDrive Driver?")
            ]}

for stream in product_QnA_agent.stream(inputs, config, stream_mode="values"):
    message=stream["messages"][-1]
    if isinstance(message, tuple):
        print(message)
    else:
        message.pretty_print()


## 03.05. Execute the Product QnA Chatbot

In [ ]:
import uuid
#Send a sequence of messages to chatbot and get its response
user_inputs = [
    "Hello",
    "I am looking to buy some golf equipment",
    "Give me a list of available products",
    "Tell me about the FairwayPro Iron Set",
    "How much does it cost?",
    "Give me similar information about the SoftSpin Tour Golf Balls",
    "Do you carry any golf bags?",
    "Thanks for the help"
]

#Create a new thread
# Limit ReAct recursion/tool loop depth to control token usage.
config = {"configurable": {"thread_id": str(uuid.uuid4())}, "recursion_limit": 6}

for input in user_inputs:
    print(f"----------------------------------------\nUSER : {input}")
    user_message = {"messages":[HumanMessage(input)]}
    ai_response = product_QnA_agent.invoke(user_message,config=config)
    print(f"AGENT : {ai_response['messages'][-1].content}")


In [ ]:
#conversation memory by user
def execute_prompt(user, config, prompt):
    inputs = {"messages":[("user",prompt)]}
    ai_response = product_QnA_agent.invoke(inputs,config=config)
    print(f"\n{user}: {ai_response['messages'][-1].content}")

#Create different session threads for 2 users
# Apply the same recursion limit for consistent token control.
config_1 = {"configurable": {"thread_id": str(uuid.uuid4())}, "recursion_limit": 6}
config_2 = {"configurable": {"thread_id": str(uuid.uuid4())}, "recursion_limit": 6}

#Test both threads
execute_prompt("USER 1", config_1, "Tell me about the StormDrive Driver")
execute_prompt("USER 2", config_2, "Tell me about the GreenLine Stand Bag")
execute_prompt("USER 1", config_1, "What is its price?")
execute_prompt("USER 2", config_2, "What is its price?")
